In [ ]:
#%pip install python-dateutil
# %pip install pandas
# %pip install numpy  
# %pip install matplotlib
# %pip install seaborn    

Limpando arquivos para o pandas conseguir ler

In [ ]:
import pandas as pd

# Abrir o arquivo CSV com codificação ANSI (windows-1252) e delimitador ';'
file_2 = 'VigiMed_Medicamentos.csv'  # Substitua pelo caminho do seu arquivo
from csv import QUOTE_NONE

skipped_lines = []

def custom_error_handler(line):
    skipped_lines.append(line)

df = pd.read_csv(
    file_2,
    encoding='windows-1252',
    sep=";",
    engine="python",
    skipinitialspace=True,
    quoting=QUOTE_NONE,
    on_bad_lines=custom_error_handler
)

# Mostrar as linhas que foram puladas
print("Linhas puladas:")
for line in skipped_lines:
    print(line)

# Exibir as primeiras linhas do DataFrame
#df.head()


Agora abrindo os 3 arquivos

In [ ]:
from csv import QUOTE_NONE

# Caminhos dos arquivos
file_1 = 'VigiMed_Notificacoes.csv' # Substitua pelo caminho do seu arquivo
file_2 = 'VigiMed_Medicamentos.csv' # Substitua pelo caminho do seu arquivo
file_3 = 'VigiMed_Reacoes.csv' # Substitua pelo caminho do seu arquivo

# Ler os arquivos CSV (originais ficarão intactos)
df1_org = pd.read_csv(file_1, encoding='windows-1252', sep=";", engine="python", skipinitialspace=True, quoting=QUOTE_NONE)
df2_org = pd.read_csv(file_2, encoding='windows-1252', sep=";", engine="python", skipinitialspace=True, quoting=QUOTE_NONE)
df3_org = pd.read_csv(file_3, encoding='windows-1252', sep=";", engine="python", skipinitialspace=True, quoting=QUOTE_NONE)

# ➜ CRIAÇÃO DAS CÓPIAS PARA TRABALHO
df1 = df1_org.copy(deep=True)
df2 = df2_org.copy(deep=True)
df3 = df3_org.copy(deep=True)

# Exibir as primeiras linhas de cada DataFrame (originais)
print("Primeiras linhas do df1:")
print(df1.head())

print("\nPrimeiras linhas do df2:")
print(df2.head())

print("\nPrimeiras linhas do df3:")
print(df3.head())



Tratamento dos arquivos

Contar campos em branco antes da limpeza

In [ ]:
#!pip install python-docx
from docx import Document

def gerar_resumo_word(df, nome_df, arquivo_word):
    """
    Cria um arquivo Word com o resumo do DataFrame:
      - nome das colunas
      - tipo
      - não nulos
      - % não nulos
    E também exibe o resumo na saída do Jupyter.
    """

    # ======== 1️⃣ Criar tabela resumo ========
    tabela_resumo = pd.DataFrame({
        "Variável": df.columns,
        "Tipo": [str(df[col].dtype) for col in df.columns],
        "Não Nulos": df.notnull().sum().values,
        "% Não Nulos": (df.notnull().sum() / len(df) * 100).round(2).values
    })

    # ======== 2️⃣ Mostrar resumo na tela ========
    print(f"\n\n📌 RESUMO DO DATAFRAME: {nome_df}")
    print("-" * 80)
    display(tabela_resumo)

    
# ======================================================
# 👉 EXECUTAR PARA df1, df2 e df3 (as cópias de trabalho)
# ======================================================

gerar_resumo_word(df1, "df1", "resumo_df1.docx")
gerar_resumo_word(df2, "df2", "resumo_df2.docx")
gerar_resumo_word(df3, "df3", "resumo_df3.docx")


In [ ]:
# import sys
# !{sys.executable} -m pip install unidecode


Tratamento pré  -  Só em DF1

In [ ]:
import numpy as np

# Padrões textuais que representam "sem informação"
padroes_none = {
    "", " ", None,
    "none", "null", "na", "n/a",
    "-", "--",
}

# Normaliza tudo para minúsculas
padroes_none = {p.lower() for p in padroes_none if isinstance(p, str)}


def padronizar_none(x):
    # Se já é NaN, mantém
    if pd.isna(x):
        return np.nan

    # Só processa strings
    if isinstance(x, str):
        s = x.strip().lower()
        if s in padroes_none or s == "":
            return np.nan
    
    return x

colunas_texto = [
    "UF", "TIPO_ENTRADA_VIGIMED", "RECEBIDO_DE",
    "TIPO_NOTIFICACAO", "SEXO", "GRAVE", "GRAVIDADE",
    "DESFECHO", "RELACAO_MEDICAMENTO_EVENTO",
    "NOME_MEDICAMENTO_WHODRUG", "ACAO_ADOTADA",
    "NOTIFICADOR", "REACAO_EVENTO_ADVERSO_MEDDRA"
]

df1_clean = df1.copy()

for col in colunas_texto:
    if col in df1_clean.columns:
        df1_clean[col] = df1_clean[col].apply(padronizar_none)


import numpy as np
import re
from datetime import datetime
DATA_CORTE = datetime(2025,11, 30)

def corrigir_data_flex(x):
    if pd.isna(x):
        return pd.NaT

    # 1. transformar em string, remover espaços
    x_clean = str(x).strip()

    # 🔧 AJUSTE MINÍMO: tirar o ".0" que vem do float
    if x_clean.endswith(".0"):
        x_clean = x_clean[:-2]

    # 2. cortar hora
    x_clean = re.sub(r"\s.*", "", x_clean)

    # 3. manter só dígitos
    num = re.sub(r"[^0-9]", "", x_clean)
    if num == "":
        return pd.NaT

    # 4. completar formatos válidos
    if len(num) == 4:            # AAAA
        num = num + "0101"
    elif len(num) == 6:          # AAAAMM
        num = num + "01"
    elif len(num) == 8:          # AAAAMMDD
        pass
    else:
        return pd.NaT            # qualquer outro formato é inválido

    # 5. tentar converter para data
    try:
        dt = datetime.strptime(num, "%Y%m%d").date()
    except:
        return pd.NaT            # datas impossíveis → NA

    # 6. eliminar datas improváveis
    if dt.year < 1910:
        return pd.NaT

    if dt > DATA_CORTE.date():
        return pd.NaT

    return dt


In [ ]:
# ============================================================
# APLICAR CORREÇÃO DE DATAS (somente estas colunas)
# ============================================================

col_datas = ["DATA_NASCIMENTO", "DATA_INICIO_HORA"]

for c in col_datas:
    if c in df1_clean.columns:
        df1_clean[c + "_FIX"] = df1_clean[c].apply(corrigir_data_flex)



print("Formato final da base limpa:", df1_clean.shape)


In [ ]:
# ============================================================
# NORMALIZAÇÃO LEVE DE TEXTO
# ============================================================
import unicodedata

def normalizar_texto(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    # remover acentos
    x = ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    )
    # remover pontuações leves
    x = re.sub(r"[^a-z0-9\s\|]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df1_clean["med_norm"] = df1_clean["NOME_MEDICAMENTO_WHODRUG"].apply(normalizar_texto)
df1_clean["evt_norm"] = df1_clean["REACAO_EVENTO_ADVERSO_MEDDRA"].apply(normalizar_texto)
df1_clean["sexo_norm"] = df1_clean["SEXO"].apply(normalizar_texto)

# ===== NORMALIZAÇÃO DAS DATAS FIX =====

# converter para datetime antes de formatar
df1_clean["DATA_NASCIMENTO_FIX"] = pd.to_datetime(df1_clean["DATA_NASCIMENTO_FIX"], errors="coerce")
df1_clean["DATA_INICIO_HORA_FIX"] = pd.to_datetime(df1_clean["DATA_INICIO_HORA_FIX"], errors="coerce")

# DATAS FIX → string "YYYY-MM-DD" ou "" se faltar
df1_clean["nasc_norm"] = df1_clean["DATA_NASCIMENTO_FIX"].dt.strftime("%Y-%m-%d")
df1_clean["nasc_norm"] = df1_clean["nasc_norm"].fillna("")

df1_clean["inicio_norm"] = df1_clean["DATA_INICIO_HORA_FIX"].dt.strftime("%Y-%m-%d")
df1_clean["inicio_norm"] = df1_clean["inicio_norm"].fillna("")


print("✔ Normalização textual aplicada sobre df1.")

# salvar versão limpa antes da deduplicação
df1_clean.to_csv("df1_clean.csv", index=False, encoding="utf-8")
print("✔ df1_clean.csv salvo.")

Deduplicar

In [ ]:
# ============================================================
# DEDUPLICAÇÃO DETERMINÍSTICA EM DOIS NÍVEIS (HIERÁRQUICA)
# Nível 1: com data de início
# Nível 2: sem data de início
# ============================================================

# ------------------------------------------------------------
# 1. CRIAR CHAVES
# ------------------------------------------------------------

# chave básica (sem data de início)
df1_clean["chave_det_basica"] = (
    df1_clean["med_norm"] + "|" +
    df1_clean["evt_norm"] + "|" +
    df1_clean["nasc_norm"] + "|" +
    df1_clean["sexo_norm"]
)

# chave com data de início
df1_clean["chave_det_inicio"] = (
    df1_clean["med_norm"] + "|" +
    df1_clean["evt_norm"] + "|" +
    df1_clean["nasc_norm"] + "|" +
    df1_clean["sexo_norm"] + "|" +
    df1_clean["inicio_norm"]
)

print("✔ Chaves determinísticas criadas.")

# ------------------------------------------------------------
# 2. NÍVEL 1 — DEDUPLICAÇÃO COM DATA DE INÍCIO
# ------------------------------------------------------------

mask_g1 = (
    (df1_clean["med_norm"] != "") &
    (df1_clean["evt_norm"] != "") &
    (df1_clean["sexo_norm"] != "") &
    (df1_clean["nasc_norm"] != "") &
    (df1_clean["inicio_norm"] != "")
)

df_g1 = df1_clean[mask_g1].copy()
df_resto = df1_clean[~mask_g1].copy()

dup_g1 = df_g1[df_g1.duplicated(subset="chave_det_inicio", keep=False)].copy()
dup_g1["nivel_dedup"] = "deterministica_com_data_inicio"
dup_g1["id_par"] = dup_g1.groupby("chave_det_inicio").ngroup() + 1

# --- selecionar um registro representativo dentro de cada grupo (nível 1) ---
# as variáveis abaixo correspondem aos componentes da chave de deduplicação
cols_n1 = ["med_norm", "evt_norm", "nasc_norm", "sexo_norm", "inicio_norm"]
df_g1["_missing_n1"] = df_g1[cols_n1].isna().sum(axis=1)

# ordena para que o mais completo (menos missing) fique primeiro em cada chave
df_g1_sorted = df_g1.sort_values(
    by=["chave_det_inicio", "_missing_n1"],
    ascending=[True, True]
)

df_g1_dedup = df_g1_sorted.drop_duplicates(subset="chave_det_inicio", keep="first").copy()
df_g1_dedup.drop(columns=["_missing_n1"], inplace=True, errors="ignore")

# (opcional) limpar coluna auxiliar do df_g1 original também, se quiser manter o objeto limpo
df_g1.drop(columns=["_missing_n1"], inplace=True, errors="ignore")

df_g1_dedup["nivel_dedup"] = "deterministica_com_data_inicio"

print(f"📌 Nível 1 — registros avaliados: {len(df_g1)}")
print(f"📌 Nível 1 — duplicatas removidas: {len(df_g1) - len(df_g1_dedup)}")

# ------------------------------------------------------------
# AUDITORIA — NÍVEL 1 (COM DATA DE INÍCIO)
# ------------------------------------------------------------

auditoria_g1 = {
    "nivel": "deterministica_com_data_inicio",
    "registros_eligiveis": len(df_g1),
    "registros_duplicados": len(dup_g1),
    "grupos_duplicidade": dup_g1["id_par"].nunique(),
    "registros_removidos": len(df_g1) - len(df_g1_dedup),
    "registros_retidos": len(df_g1_dedup),
}

auditoria_g1

# ------------------------------------------------------------
# 3. NÍVEL 2 — DEDUPLICAÇÃO SEM DATA DE INÍCIO
# ------------------------------------------------------------

mask_g2 = (
    (df_resto["med_norm"] != "") &
    (df_resto["evt_norm"] != "") &
    (df_resto["sexo_norm"] != "") &
    (df_resto["nasc_norm"] != "")
)

df_g2 = df_resto[mask_g2].copy()
df_incompleta = df_resto[~mask_g2].copy()

dup_g2 = df_g2[df_g2.duplicated(subset="chave_det_basica", keep=False)].copy()
dup_g2["nivel_dedup"] = "deterministica_sem_data_inicio"
dup_g2["id_par"] = dup_g2.groupby("chave_det_basica").ngroup() + 1



# --- selecionar um registro representativo dentro de cada grupo (nível 2) ---
# as variáveis abaixo correspondem aos componentes da chave de deduplicação
cols_n2 = ["med_norm", "evt_norm", "nasc_norm", "sexo_norm"]
df_g2["_missing_n2"] = df_g2[cols_n2].isna().sum(axis=1)

df_g2_sorted = df_g2.sort_values(
    by=["chave_det_basica", "_missing_n2"],
    ascending=[True, True]
)

df_g2_dedup = df_g2_sorted.drop_duplicates(subset="chave_det_basica", keep="first").copy()
df_g2_dedup.drop(columns=["_missing_n2"], inplace=True, errors="ignore")
df_g2.drop(columns=["_missing_n2"], inplace=True, errors="ignore")

df_g2_dedup["nivel_dedup"] = "deterministica_sem_data_inicio"

df_incompleta["nivel_dedup"] = "nao_deduplicado"
# ------------------------------------------------------------
# NÍVEL 0 — REMOVER DUPLICATAS EXATAS (LINHAS IDÊNTICAS) EM INCOMPLETAS
# ------------------------------------------------------------
n_incompleta_antes = len(df_incompleta)

# remove linhas 100% idênticas (todas as colunas)
df_incompleta = df_incompleta.drop_duplicates(keep="first").copy()

n_incompleta_depois = len(df_incompleta)
print(f"📌 Nível 0 — incompletas antes: {n_incompleta_antes}")
print(f"📌 Nível 0 — duplicatas exatas removidas: {n_incompleta_antes - n_incompleta_depois}")


print(f"📌 Nível 2 — registros avaliados: {len(df_g2)}")
print(f"📌 Nível 2 — duplicatas removidas: {len(df_g2) - len(df_g2_dedup)}")

# ------------------------------------------------------------
# AUDITORIA — NÍVEL 2 (SEM DATA DE INÍCIO)
# ------------------------------------------------------------

auditoria_g2 = {
    "nivel": "deterministica_sem_data_inicio",
    "registros_eligiveis": len(df_g2),
    "registros_duplicados": len(dup_g2),
    "grupos_duplicidade": dup_g2["id_par"].nunique(),
    "registros_removidos": len(df_g2) - len(df_g2_dedup),
    "registros_retidos": len(df_g2_dedup),
}

auditoria_g2

#AUDITORIA — NÍVEL 0 (DUPLICATAS EXATAS EM INCOMPLETAS)#

auditoria_g0 = {
    "nivel": "duplicatas_exatas_incompletas",
    "registros_antes": n_incompleta_antes,
    "registros_removidos": n_incompleta_antes - n_incompleta_depois,
    "registros_retidos": n_incompleta_depois,
}
auditoria_g0

# ------------------------------------------------------------
# 4. CONSOLIDAR BASE FINAL
# ------------------------------------------------------------

df1_dedup = pd.concat(
    [df_g1_dedup, df_g2_dedup, df_incompleta],
    ignore_index=True
)

print("✔ Base final consolidada.")
print("📌 Total final:", len(df1_dedup))

# ------------------------------------------------------------
# 5. EXPORTAR AUDITORIA DE DUPLICATAS
# ------------------------------------------------------------

dup_final = pd.concat([dup_g1, dup_g2], ignore_index=True)
dup_final = dup_final.sort_values(by=["nivel_dedup", "id_par"])


print("📁 Arquivo gerado: df1_pares_duplicados_dois_niveis.csv")
print("🔢 Total de grupos de duplicidade:", dup_final["id_par"].nunique())
print("🧬 Total de linhas duplicadas registradas:", len(dup_final))

# ============================================================
# 6. AUDITORIA FINAL CONSOLIDADA (ANTES / DEPOIS)
# ============================================================

total_inicial = len(df1_clean)
total_final = len(df1_dedup)

total_removidos = total_inicial - total_final
perc_removidos = (total_removidos / total_inicial) * 100
perc_retidos = (total_final / total_inicial) * 100

# detalhamento por nível
removidos_n1 = len(df_g1) - len(df_g1_dedup)
removidos_n2 = len(df_g2) - len(df_g2_dedup)
removidos_n0 = n_incompleta_antes - n_incompleta_depois

total_removidos_calc = removidos_n0 + removidos_n1 + removidos_n2

print("🔎 Checagem removidos:")
print(" - Nível 0:", removidos_n0)
print(" - Nível 1:", removidos_n1)
print(" - Nível 2:", removidos_n2)
print(" - Soma por níveis:", total_removidos_calc)
print(" - Diferença total (inicial - final):", total_removidos)

auditoria_resumo = pd.DataFrame([
    {
        "etapa": "Total inicial",
        "registros": total_inicial,
        "percentual": 100.0
    },
    {
        "etapa": "Removidos – nível 1 (com data de início)",
        "registros": removidos_n1,
        "percentual": (removidos_n1 / total_inicial) * 100
    },
    {
        "etapa": "Removidos – nível 2 (sem data de início)",
        "registros": removidos_n2,
        "percentual": (removidos_n2 / total_inicial) * 100
    },
    {
        "etapa": "Total removido (deduplicação)",
        "registros": total_removidos,
        "percentual": perc_removidos
    },
    {
        "etapa": "Total final (deduplicado)",
        "registros": total_final,
        "percentual": perc_retidos
    },
    {
    "etapa": "Removidos – nível 0 (duplicatas exatas em incompletas)",
    "registros": removidos_n0,
    "percentual": (removidos_n0 / total_inicial) * 100
    }
])

auditoria_resumo

# ============================================================
# 7. EXPORTAÇÃO (MANTENDO OS ARQUIVOS DO SEU PADRÃO ORIGINAL)
# - df1_dedup.csv (base final deduplicada)
# - df1_duplicadas.csv (todas as duplicatas, empilhadas)
# - df1_pares_duplicados.csv (mesmo conceito do seu arquivo original)
# ============================================================

# 7.1 Base final deduplicada
df1_dedup.to_csv("df1_dedup.csv", index=False, encoding="utf-8")

# 7.2 Todas as duplicatas (nível 1 + nível 2) empilhadas
df1_duplicadas = pd.concat([dup_g1, dup_g2], ignore_index=True)

# IMPORTANTE: para garantir id_par único entre níveis
dup_g2_offset = dup_g1["id_par"].nunique()
df1_duplicadas.loc[df1_duplicadas["nivel_dedup"] == "deterministica_sem_data_inicio", "id_par"] = (
    df1_duplicadas.loc[df1_duplicadas["nivel_dedup"] == "deterministica_sem_data_inicio", "id_par"] + dup_g2_offset
)

df1_duplicadas = df1_duplicadas.sort_values(by=["nivel_dedup", "id_par"])

df1_duplicadas.to_csv("df1_duplicadas.csv", index=False, encoding="utf-8")

# 7.3 Arquivo “pares duplicados” no MESMO padrão da sua versão original
# (linhas empilhadas, com id_par e ordenação para revisão manual)
df1_pares_duplicados = df1_duplicadas.copy()
df1_pares_duplicados = df1_pares_duplicados.sort_values(by=["id_par", "nivel_dedup"])

df1_pares_duplicados.to_csv("df1_pares_duplicados.csv", index=False, encoding="utf-8")

print("✔ Arquivos exportados (padrão mantido):")
print("- df1_dedup.csv (deduplicado final)")
print("- df1_duplicadas.csv (todas as duplicatas empilhadas)")
print("- df1_pares_duplicados.csv (pares/grupos empilhados para revisão manual)")
print("🔢 Grupos (id_par) no arquivo de pares:", df1_pares_duplicados["id_par"].nunique())
print("🧬 Linhas no arquivo de pares:", len(df1_pares_duplicados))


In [ ]:
#!pip install python-docx

def gerar_resumo_word(df, nome_df, arquivo_word):
    """
    Cria um arquivo Word com o resumo do DataFrame:
      - nome das colunas
      - tipo
      - não nulos
      - % não nulos
    E também exibe o resumo na saída do Jupyter.
    """

    # ======== 1️⃣ Criar tabela resumo ========
    tabela_resumo = pd.DataFrame({
        "Variável": df.columns,
        "Tipo": [str(df[col].dtype) for col in df.columns],
        "Não Nulos": df.notnull().sum().values,
        "% Não Nulos": (df.notnull().sum() / len(df) * 100).round(2).values
    })

    # ======== 2️⃣ Mostrar resumo na tela ========
    print(f"\n\n📌 RESUMO DO DATAFRAME: {nome_df}")
    print("-" * 80)
    display(tabela_resumo)

    # ======== 3️⃣ Gerar arquivo Word ========
    doc = Document()
    doc.add_heading(f"Resumo das Variáveis – {nome_df}", level=1)
    doc.add_paragraph(f"Total de linhas: {len(df):,}")

    tabela = doc.add_table(rows=1, cols=len(tabela_resumo.columns))
    tabela.style = 'Table Grid'

    # Cabeçalhos
    hdr_cells = tabela.rows[0].cells
    for i, coluna in enumerate(tabela_resumo.columns):
        hdr_cells[i].text = coluna

    # Linhas
    for _, row in tabela_resumo.iterrows():
        linha = tabela.add_row().cells
        linha[0].text = str(row["Variável"])
        linha[1].text = str(row["Tipo"])
        linha[2].text = str(row["Não Nulos"])
        linha[3].text = str(row["% Não Nulos"])



# ======================================================
# 👉 EXECUTAR PARA df1_dedup (as cópias de trabalho)
# ======================================================

gerar_resumo_word(df1_dedup, "df1_dedup", "resumo_df1_DEDUP.docx")


Unir as tabelas de Notificações e Medicamentos com limpeza de DF2

In [ ]:
#### Limpeza de duplicats em DF2#########

# Criar coluna com número de campos não nulos por linha
df2["n_nonnull"] = df2.notna().sum(axis=1)

# Ordenar: primeiro por chave, depois por "n_nonnull" de forma decrescente
df2_sorted = df2.sort_values(
    by=["IDENTIFICACAO_NOTIFICACAO",
        "RELACAO_MEDICAMENTO_EVENTO",
        "NOME_MEDICAMENTO_WHODRUG",
        "n_nonnull"],
    ascending=[True, True, True, False]
)

# Deduplicar mantendo apenas o mais completo
df2_clean = df2_sorted.drop_duplicates(
    subset=["IDENTIFICACAO_NOTIFICACAO",
            "RELACAO_MEDICAMENTO_EVENTO",
            "NOME_MEDICAMENTO_WHODRUG"],
    keep="first"
)

# Remover a coluna auxiliar
df2_clean = df2_clean.drop(columns=["n_nonnull"])

print("Antes:", df2.shape)
print("Depois:", df2_clean.shape)


In [ ]:
# Realizar o merge entre df1_DEDUP e df2_clean
merged_df = pd.merge(df1_dedup, df2_clean, on='IDENTIFICACAO_NOTIFICACAO', how='left')

# Exibir as primeiras linhas do DataFrame resultante
#merged_df.head()


In [ ]:
# Verificar duplicatas completas (todas as colunas)
dup_full = merged_df[merged_df.duplicated(keep=False)]

print("Total de duplicatas de linha completa:", len(dup_full))
dup_full.head()


Combinar tabela Reações - DF3

In [ ]:
# Realizar limpeza de Df3

# Criar coluna auxiliar com número de campos não nulos
df3["n_nonnull"] = df3.notna().sum(axis=1)

# Ordenar: primeiro por chave, depois por completude
df3_sorted = df3.sort_values(
    by=["IDENTIFICACAO_NOTIFICACAO", "PT", "n_nonnull"],
    ascending=[True, True, False]
)

# Deduplicação por ID + PT, mantendo o mais completo
df3_clean = df3_sorted.drop_duplicates(
    subset=["IDENTIFICACAO_NOTIFICACAO", "PT"],
    keep="first"
).drop(columns=["n_nonnull"])

print("Antes:", df3.shape)
print("Depois:", df3_clean.shape)

In [ ]:
#Merged Completo

# Merge final: cada medicamento será combinado a cada reação da mesma ID
final_df = pd.merge(
    merged_df,       # resultado do merge df1_dedup × df2_clean
    df3_clean, 
    on="IDENTIFICACAO_NOTIFICACAO",
    how="left"
)

print("Formato final do merge (df1_dedup + df2_clean + df3_clean):", final_df.shape)
final_df.head()


In [ ]:
dup_full = final_df[final_df.duplicated(keep=False)]

print("Total de duplicatas de linha completa no merge final:", len(dup_full))

# Detecta NA, string vazia e espaços
mask_pt_vazio = (
    final_df["PT"].isna() |
    (final_df["PT"].astype(str).str.strip() == "")
)

print("Total de registros com PT vazio ou NA:", mask_pt_vazio.sum())

# Inspecionar
final_df[mask_pt_vazio].head()

campo_med = "PRINCIPIOS_ATIVOS_WHODRUG"   # ajuste se necessário

mask_med_vazio = (
    final_df[campo_med].isna() |
    (final_df[campo_med].astype(str).str.strip() == "")
)

print("Total de registros com princípio ativo vazio ou NA:", mask_med_vazio.sum())

final_df[mask_med_vazio].head()

mask_critico = mask_pt_vazio & mask_med_vazio

print("Registros sem medicamento E sem PT:", mask_critico.sum())
final_df[mask_critico].head()


In [ ]:
mask_feto = (
    final_df["GRUPO_IDADE"]
        .astype(str)
        .str.strip()
        .str.lower() == "feto"
)

print("Total de registros classificados como Feto:", mask_feto.sum())


In [ ]:
# Número de linhas antes da exclusão
linhas_antes = len(final_df)

# Criar máscara crítica (PT vazio, princípio ativo vazio, E FETO)
mask_critico = (
    final_df["PT"].isna() |
    (final_df["PT"].astype(str).str.strip() == "") |
    final_df["PRINCIPIOS_ATIVOS_WHODRUG"].isna() |
    (final_df["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.strip() == "") |
    (final_df["GRUPO_IDADE"].astype(str).str.strip().str.lower() == "feto")
)

# Aplicar exclusão
final_df_limpo = final_df[~mask_critico].copy()

# Número de linhas depois da exclusão
linhas_depois = len(final_df_limpo)

print(f"Linhas antes da exclusão: {linhas_antes}")
print(f"Linhas excluídas: {linhas_antes - linhas_depois}")
print(f"Linhas restantes após exclusão: {linhas_depois}")

final_df_limpo.to_csv("final_df_limpo_1225.csv", index=False, encoding="utf-8")
print(final_df_limpo.columns.tolist())



# Filtrar apenas medicamentos suspeitos e não administrados
#RELACAO_MEDICAMENTO_EVENTO_x vem de df1 (só a primeira linha da notificação) - usar RELACAO_MEDICAMENTO_EVENTO_y#)

In [ ]:



final_df_sus = final_df_limpo[
    final_df_limpo["RELACAO_MEDICAMENTO_EVENTO_y"].str.lower() == "suspeito"
]

# Contar número de medicamentos suspeitos por notificação
contagem_suspeitos = (
    final_df_sus.groupby("IDENTIFICACAO_NOTIFICACAO")["PRINCIPIOS_ATIVOS_WHODRUG"]
           .nunique()
           .reset_index(name="n_suspeitos")
)

# Distribuição: quantos têm 1, 2, 3, 4... suspeitos
dist_suspeitos = contagem_suspeitos["n_suspeitos"].value_counts().sort_index()

print(dist_suspeitos)

dist_suspeitos = contagem_suspeitos["n_suspeitos"].value_counts().sort_index()

import numpy as np

valores = contagem_suspeitos["n_suspeitos"]

media = valores.mean()
mediana = valores.median()
desvio = valores.std()   # desvio padrão amostral
minimo = valores.min()
maximo = valores.max()

print(f"Média: {media:.2f}")
print(f"Mediana: {mediana}")
print(f"Desvio-padrão: {desvio:.2f}")
print(f"Mínimo: {minimo}")
print(f"Máximo: {maximo}")

estatisticas = {
    "Média": media,
    "Mediana": mediana,
    "Desvio-padrão": desvio,
    "Mínimo": minimo,
    "Máximo": maximo
}

print(estatisticas)


In [ ]:
final_df_nao_adm = final_df_limpo[
    final_df_limpo["RELACAO_MEDICAMENTO_EVENTO_y"] == "Medicamento não administrado"
]

contagem_nao_adm = (
    final_df_nao_adm.groupby("IDENTIFICACAO_NOTIFICACAO")["PRINCIPIOS_ATIVOS_WHODRUG"]
               .nunique()
               .reset_index(name="n_nao_admin")
)

dist_nao = contagem_nao_adm["n_nao_admin"].value_counts().sort_index()

print(dist_nao)

dist_nao = contagem_nao_adm["n_nao_admin"].value_counts().sort_index()

import numpy as np

valores = contagem_nao_adm["n_nao_admin"]

media = valores.mean()
mediana = valores.median()
desvio = valores.std()   # desvio padrão amostral
minimo = valores.min()
maximo = valores.max()

print(f"Média: {media:.2f}")
print(f"Mediana: {mediana}")
print(f"Desvio-padrão: {desvio:.2f}")
print(f"Mínimo: {minimo}")
print(f"Máximo: {maximo}")

estatisticas = {
    "Média": media,
    "Mediana": mediana,
    "Desvio-padrão": desvio,
    "Mínimo": minimo,
    "Máximo": maximo
}

print(estatisticas)

Gerar um df só com casos suspeitos e não administrados

In [ ]:
df_trabalho_suspeito_nao_adm = final_df_limpo[
    final_df_limpo["RELACAO_MEDICAMENTO_EVENTO_y"]
        .isin(["Suspeito", "Medicamento não administrado"])
].copy()

df_trabalho_suspeito_nao_adm.to_csv("df_trabalho_suspeito_nao_adm_y.csv", index=False, encoding="utf-8")

linhas, colunas = df_trabalho_suspeito_nao_adm.shape
print(f"Linhas: {linhas}, Colunas: {colunas}")



In [ ]:
# Caminho para o arquivo harmonizacao.csv
harmonizacao_file = 'harmonizacao_2.csv'


# Ler o arquivo harmonizacao.csv
harmonizacao_df = pd.read_csv(harmonizacao_file, encoding='windows-1252', sep=';')

# Exibir as primeiras linhas do DataFrame
#print(harmonizacao_df.head())
print(harmonizacao_df.columns)


In [ ]:
# Verificar as linhas duplicadas em df_trabalho_suspeito antes do merge
duplicated_rows_before = df_trabalho_suspeito_nao_adm[df_trabalho_suspeito_nao_adm.duplicated(subset=df_trabalho_suspeito_nao_adm.columns, keep=False)]
print(f"Número de linhas duplicadas em df_trabalho_suspeito antes do merge: {duplicated_rows_before.shape[0]}")
print("Exemplos de duplicatas antes do merge:")
print(duplicated_rows_before.head())

# Realizar o merge entre df_trabalho_suspeito e harmonizacao_df
df_trabalho_suspeito_final = df_trabalho_suspeito_nao_adm.merge(
    harmonizacao_df,
    on='PRINCIPIOS_ATIVOS_WHODRUG',
    how='left',
)

# Verificar as linhas duplicadas no resultado do merge
duplicated_rows = df_trabalho_suspeito_final[df_trabalho_suspeito_final.duplicated(subset=df_trabalho_suspeito_nao_adm.columns, keep=False)]

print(f"Número de linhas duplicadas após o merge: {duplicated_rows.shape[0]}")
print("Exemplos de duplicatas:")
print(duplicated_rows.head())

# Excluir as linhas duplicadas no resultado do merge
df_trabalho_suspeito_final = df_trabalho_suspeito_final.drop_duplicates()

# Verificar novamente o número de linhas duplicadas após a exclusão
duplicated_rows_after = df_trabalho_suspeito_final[df_trabalho_suspeito_final.duplicated(subset=df_trabalho_suspeito_nao_adm.columns, keep=False)]
print(f"Número de linhas duplicadas após a exclusão: {duplicated_rows_after.shape[0]}")

# Preencher valores ausentes na coluna Harmonização com os valores originais de PRINCIPIOS_ATIVOS_WHODRUG
df_trabalho_suspeito_final['Harmonização'] = df_trabalho_suspeito_final['Harmonização'].fillna(df_trabalho_suspeito_final['PRINCIPIOS_ATIVOS_WHODRUG'])


print(f"Número de linhas no df_trabalho_suspeito_nao_adm_y_final_1225: {df_trabalho_suspeito_final.shape[0]}")

Harmonização da idade

In [ ]:
#apagar colunas _norm criadas na deduplicação e manutenção das _FIX
df_trabalho_suspeito_final = df_trabalho_suspeito_final.loc[
    :, 
    ~df_trabalho_suspeito_final.columns.str.endswith('_norm')
]

print(df_trabalho_suspeito_final.columns)

In [ ]:

# Copia o DataFrame original para um novo
dados_com_idade_suspeito_df = df_trabalho_suspeito_final.copy()

#corrigir data inicio, que não havia sido corrigida na deduplicação
dados_com_idade_suspeito_df["INICIO_ADMINISTRACAO_FIX"] = (
    dados_com_idade_suspeito_df["INICIO_ADMINISTRACAO"]
        .apply(corrigir_data_flex)
)

dados_com_idade_suspeito_df["INICIO_ADMINISTRACAO_FIX"] = pd.to_datetime(
    dados_com_idade_suspeito_df["INICIO_ADMINISTRACAO_FIX"],
    errors="coerce"
)


from datetime import datetime


# Função reutilizável para calcular idade em anos decimais
def calcular_idade_decimal_com_base(nascimento, referencia):
    if pd.notnull(nascimento) and pd.notnull(referencia):
        if nascimento.year < 1910 or referencia > pd.Timestamp(DATA_CORTE):
            return None
        
        dias = (referencia - nascimento).days
        idade_anos = dias / 365.25
        
        if idade_anos < 0 or idade_anos > 130:
            return None
        
        return round(idade_anos, 2)
    else:
        return None


# Aplica a função para DATA_INICIO_HORA_FIX
dados_com_idade_suspeito_df['IDADE'] = dados_com_idade_suspeito_df.apply(
    lambda row: calcular_idade_decimal_com_base(
        row['DATA_NASCIMENTO_FIX'],
        row['DATA_INICIO_HORA_FIX']
    ),
    axis=1
)

# Aplica a função para INICIO_ADMINISTRACAO_FIX
dados_com_idade_suspeito_df['IDADE_1'] = dados_com_idade_suspeito_df.apply(
    lambda row: calcular_idade_decimal_com_base(
        row['DATA_NASCIMENTO_FIX'],
        row['INICIO_ADMINISTRACAO_FIX']
    ),
    axis=1
)

# Visualização
print(
    dados_com_idade_suspeito_df[
        ['DATA_NASCIMENTO_FIX',
         'INICIO_ADMINISTRACAO_FIX',
         'DATA_INICIO_HORA_FIX',
         'IDADE',
         'IDADE_1']
    ].head(10)
)


In [ ]:
# Selecionar apenas valores não nulos e converter para string
idade_txt = (
    dados_com_idade_suspeito_df['IDADE_MOMENTO_REACAO']
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
)

# Extrair o texto que vem depois do número
# Ex: "10 anos" -> "anos"
#     "2 meses" -> "meses"
#     "3 decadas" -> "decadas"
sufixos = idade_txt.str.extract(
    r'\d+\s*([a-zçãõêéíóúô\s]+)',
    expand=False
)

# Limpar espaços extras
sufixos = sufixos.str.strip()

# Visualizar exemplos
#sufixos.head(20)

mapa_sufixos = (
    sufixos
        .value_counts(dropna=False)
        .reset_index()
        .rename(columns={
            'index': 'sufixo_textual',
            'count': 'frequencia'
        })
)

mapa_sufixos



In [ ]:
def texto_para_idade_decimal(texto):
    if pd.isnull(texto):
        return None
    
    texto = str(texto).lower().strip()

    # Extrair valor numérico
    match_valor = re.search(r'(\d+)', texto)
    if not match_valor:
        return None
    
    valor = int(match_valor.group(1))

    # Identificar unidade e converter para anos
    if 'ano' in texto:
        idade_anos = valor

    elif 'mês' in texto or 'mes' in texto:
        idade_anos = valor / 12

    elif 'semana' in texto:
        idade_anos = valor / 52.1429

    elif 'dia' in texto:
        idade_anos = valor / 365.25

    elif 'hora' in texto:
        idade_anos = valor / (365.25 * 24)

    elif 'década' in texto or 'decada' in texto:
        idade_anos = valor * 10

    else:
        return None  # unidade não reconhecida

    # Filtro de plausibilidade (coerente com o resto do pipeline)
    if idade_anos < 0 or idade_anos > 130:
        return None

    return round(idade_anos, 2)

dados_com_idade_suspeito_df['IDADE_2'] = (
    dados_com_idade_suspeito_df['IDADE_MOMENTO_REACAO']
        .apply(texto_para_idade_decimal)
)

print(
    dados_com_idade_suspeito_df[
        ['IDADE_MOMENTO_REACAO', 'IDADE_2']
    ].head(10)
)



In [ ]:

def comparar_idades(row):
    idade = row['IDADE']
    idade_2 = row['IDADE_2']
    idade_1 = row['IDADE_1']

    # Decide qual valor usar (prioridade: IDADE > IDADE_2 > IDADE_1)
    if pd.notnull(idade):
        val = idade
    elif pd.notnull(idade_2):
        val = idade_2
    elif pd.notnull(idade_1):
        val = idade_1
    else:
        return None

    # Garante que retorna um número com 2 casas decimais
    return round(float(val), 2)

# Aplica a função para gerar a nova coluna
dados_com_idade_suspeito_df['IDADE_FINAL'] = dados_com_idade_suspeito_df.apply(comparar_idades, axis=1)

# Garante que a coluna é do tipo float
dados_com_idade_suspeito_df['IDADE_FINAL'] = pd.to_numeric(dados_com_idade_suspeito_df['IDADE_FINAL'], errors='coerce')

# Opcional: criar uma coluna separada para identificar divergências
dados_com_idade_suspeito_df['DIVERGENCIA'] = dados_com_idade_suspeito_df.apply(
    lambda row: 'DIFER' if (
        pd.notnull(row['IDADE']) and pd.notnull(row['IDADE_2']) and
        abs(row['IDADE'] - row['IDADE_2']) >= 0.99
    ) else '', axis=1
)

# Visualiza os resultados
print(dados_com_idade_suspeito_df[['IDADE', 'IDADE_2', 'IDADE_1', 'IDADE_FINAL', 'DIVERGENCIA']].head(10))
print(dados_com_idade_suspeito_df['IDADE_FINAL'].dtype)  # Confirma o tipo da coluna




Gerar faixas etárias

In [ ]:
import pandas as pd

# Função para classificar a idade
def classificar_faixa_etaria(idade):
    if pd.isnull(idade):
        return 'Ignorado'
    
    dias = idade * 365.25  # Converte idade em anos para dias
    
    
    if dias <= 30:
        return 'Neonato (0-30 dias)'
    elif idade <= 5:
        return 'Infantil (31 dias - 5 anos)'
    elif idade <= 12:
        return 'Criança (6-12 anos)'
    elif idade <= 18:
        return 'Adolescente (13-18 anos)'
    elif idade < 65:
        return 'Adulto (19-64 anos)'
    else:
        return 'Idoso (65+ anos)'


# Aplica a função e cria nova coluna
dados_com_idade_suspeito_df['FAIXA_ETARIA'] = dados_com_idade_suspeito_df['IDADE_FINAL'].apply(classificar_faixa_etaria)

# Visualiza o resultado
print(dados_com_idade_suspeito_df[['IDADE_FINAL', 'FAIXA_ETARIA']].head(10))

# Mostra o quantitativo de campos 'Ignorado' na coluna FAIXA_ETARIA
num_ignorados = (dados_com_idade_suspeito_df['FAIXA_ETARIA'].astype(str).str.strip() == 'Ignorado').sum()
print(f"Quantidade de campos 'Ignorado' em FAIXA_ETARIA: {num_ignorados}")



Mesclar com a coluna 'GRUPO_IDADE'

In [ ]:
# Dicionário de mapeamento para uniformizar nomenclatura das faixas etárias
faixas_map = {
    'Adulto': 'Adulto (19-64 anos)',
    'Idoso': 'Idoso (65+ anos)',
    'Criança': 'Criança (6-12 anos)',
    'Infantil': 'Infantil (31 dias - 5 anos)',
    'Adolescente': 'Adolescente (13-18 anos)',
    'Neonato': 'Neonato (0-30 dias)'
}

# 1️⃣ Criar a coluna GRUPO_IDADE_FIX a partir da original
dados_com_idade_suspeito_df['GRUPO_IDADE_FIX'] = dados_com_idade_suspeito_df['GRUPO_IDADE']

# 2️⃣ Aplicar a padronização na coluna FIX
dados_com_idade_suspeito_df['GRUPO_IDADE_FIX'] = (
    dados_com_idade_suspeito_df['GRUPO_IDADE_FIX']
        .replace(faixas_map)
)

# Conferência dos valores únicos já padronizados
print(dados_com_idade_suspeito_df['GRUPO_IDADE_FIX'].dropna().unique())

# 3️⃣ Preencher FAIXA_ETARIA apenas se estiver 'Ignorado'
dados_com_idade_suspeito_df['FAIXA_ETARIA'] = dados_com_idade_suspeito_df.apply(
    lambda row: row['GRUPO_IDADE_FIX']
    if str(row['FAIXA_ETARIA']).strip() == 'Ignorado'
       and pd.notnull(row['GRUPO_IDADE_FIX'])
       and str(row['GRUPO_IDADE_FIX']).strip() != ''
    else row['FAIXA_ETARIA'],
    axis=1
)

# Visualização
print(dados_com_idade_suspeito_df[['IDADE_FINAL', 'FAIXA_ETARIA', 'GRUPO_IDADE_FIX']].head(10))

# Quantitativo de 'Ignorado' remanescentes
num_ignorados = (dados_com_idade_suspeito_df['FAIXA_ETARIA'].astype(str).str.strip() == 'Ignorado').sum()
print(f"Quantidade de campos 'Ignorado' em FAIXA_ETARIA: {num_ignorados}")

print(dados_com_idade_suspeito_df['FAIXA_ETARIA'].dropna().unique())



In [ ]:
def gerar_resumo_word(df, nome_df, arquivo_word):
    """
    Cria um arquivo Word com o resumo do DataFrame:
      - nome das colunas
      - tipo
      - não nulos
      - % não nulos
    E também exibe o resumo na saída do Jupyter.
    """

    # ======== 1️⃣ Criar tabela resumo ========
    tabela_resumo = pd.DataFrame({
        "Variável": df.columns,
        "Tipo": [str(df[col].dtype) for col in df.columns],
        "Não Nulos": df.notnull().sum().values,
        "% Não Nulos": (df.notnull().sum() / len(df) * 100).round(2).values
    })

    # ======== 2️⃣ Mostrar resumo na tela ========
    print(f"\n\n📌 RESUMO DO DATAFRAME: {nome_df}")
    print("-" * 80)
    display(tabela_resumo)

    
# ======================================================
# 👉 EXECUTAR 
# ======================================================

gerar_resumo_word(dados_com_idade_suspeito_df, "dados_com_idade_suspeito_nao_adm_y_df", "resumo_dados_com_idade_suspeito_nao_adm_y_df.docx")

Filtrar J01 

In [ ]:
#Filtrar as linhas que possuem "J01" em qualquer posição na coluna PRINCIPIOS_ATIVOS_WHODRUG
# Filtrar as linhas que possuem "J01" em qualquer posição na coluna CODIGO_ATC
filtered_j01_suspeito = dados_com_idade_suspeito_df[dados_com_idade_suspeito_df['CODIGO_ATC'].str.contains('J01', na=False)]

#USAR SE FOR EXCLUIR ANTES # Lista de padrões/exclusões (case insensitive)
#excluídos princípios ativos com classificação ATC múltipla que não representam antibacterianos sistêmicos"#
exclusoes = [
    'Nitrato de cerio|Sulfadiazine',
    'D-mannose|Vaccinium macrocarpon',
    'Vaccinium spp. fruit',
    'Tribulus terrestris',
    'Benzoilmetronidazol',
    'Rifampicin',
    'Secnidazole',
    'Bacitracin',
    'Linum usitatissimum seed',
    'Allium sativum oil',
    'Sulfato de neomicina' 
]

# Excluir linhas onde 'Harmonização' contém qualquer um dos padrões de exclusão
for padrao in exclusoes:
    filtered_j01_suspeito = filtered_j01_suspeito[~filtered_j01_suspeito['Harmonização'].str.contains(padrao, case=False, na=False)]

print(f"Número de linhas no DataFrame filtrado J01: {filtered_j01_suspeito.shape[0]}")
# Salvar o DataFrame filtrado em um arquivo CSV
filtered_j01_suspeito.to_csv('filtered_j01_suspeito_nao_adm_y_analise_1225.csv', index=False, encoding='utf-8')
print("Arquivo 'filtered_j01_suspeito_nao_adm_y_analise_1225.csv' salvo com sucesso!")

## Comentado, pois não é mais necessário para o código atual, apenas em caso de mudanças em Harmonização de PAs
# Obter os valores únicos da coluna PRINCIPIOS_ATIVOS_WHODRUG após o filtro
unique_principios_j01 = filtered_j01_suspeito['Harmonização'].unique()
unique_ID_j01 = filtered_j01_suspeito['IDENTIFICACAO_NOTIFICACAO'].unique()


# Printar a lista de valores únicos
print(f"Número de Antibioticos no DataFrame filtrado J01: {unique_principios_j01.shape[0]}")
print(f"Número de Id unicas no DataFrame filtrado J01: {unique_ID_j01.shape[0]}")

In [ ]:
# Listar PTs para J01 com contagem

# --------------------------------------------------
# Colunas MedDRA esperadas
# --------------------------------------------------
cols_meddra = ["SOC", "HLGT", "HLT", "PT", "REACAO_EVTO_ADVERSO_MEDDRA_LLT"]

# --------------------------------------------------
# Selecionar apenas colunas relevantes
# --------------------------------------------------
df_meddra_j01 = (
    filtered_j01_suspeito[cols_meddra]
    .dropna(subset=["REACAO_EVTO_ADVERSO_MEDDRA_LLT"])
    .astype(str)
)

# Remover espaços residuais
for col in cols_meddra:
    df_meddra_j01[col] = df_meddra_j01[col].str.strip()

# --------------------------------------------------
# 1) Contagem de ocorrências por PT
# --------------------------------------------------
pt_counts = (
    df_meddra_j01["REACAO_EVTO_ADVERSO_MEDDRA_LLT"]
    .value_counts()
    .rename("n_ocorrencias")
    .reset_index()
    .rename(columns={"index": "REACAO_EVTO_ADVERSO_MEDDRA_LLT"})
)

# --------------------------------------------------
# 2) Agregar para garantir 1 linha por PT
# --------------------------------------------------
df_pts_meddra = (
    df_meddra_j01
    .drop_duplicates(subset=["REACAO_EVTO_ADVERSO_MEDDRA_LLT"])
    .merge(pt_counts, on="REACAO_EVTO_ADVERSO_MEDDRA_LLT", how="left")
    .sort_values(by=["SOC", "HLGT", "HLT", "PT", "REACAO_EVTO_ADVERSO_MEDDRA_LLT"])
    .reset_index(drop=True)
)

# --------------------------------------------------
# Salvar CSV
# --------------------------------------------------
df_pts_meddra.to_csv(
    "LLTs_unicos_J01_com_MedDRA_n_susp_nao_adm.csv",
    index=False,
    encoding="utf-8"
)

print(f"CSV salvo com {len(df_pts_meddra)} LLTs únicos (J01) com SOC/HLGT/HLT e contagem.")


Formação dos agrupamento de MedDRas de interesse

Filtro SMQ MedDRA Erro de Medicação

In [ ]:

# =========================
# 1) Carregar a lista de PTs do Excel
#"IMPORTANTE: Ajuste o caminho para o arquivo Excel baixado
#DO MedDRA, SMQ Medication Error, LINGUAGEM: Brazilian VERSÃO: 28.1"
# =========================


caminho_excel = r"Pts smq erro de medicação.xlsx"
# ajuste o caminho para o arquivo excel baixado 

smq_df = pd.read_excel(caminho_excel)

col_smq = "SMQ Medication Error Brazilian 28.1"  # coluna A

# Conjunto de PTs exatamente como estão no Excel
smq_set = set(
    smq_df[col_smq]
    .dropna()
    .astype(str)
)

# =========================
# 2) Filtrar no dataframe principal
# =========================
filtered_j01_suspeito = filtered_j01_suspeito.copy()

# Ajuste aqui se o nome da coluna de PT no seu dataframe principal for diferente
col_pt = "PT"

# Filtrar apenas os PTs que estão na lista do Excel
df_pts_smq_medication_error = filtered_j01_suspeito[
    filtered_j01_suspeito[col_pt].isin(smq_set)
].copy()

# =========================
# 3) Criar coluna de grupamento
# =========================
df_pts_smq_medication_error["Grupamento"] = "SMQ Erro de medicacao"


# =========================
# 4) Visualizar resultado
# ========================
print("Total de linhas filtradas:", len(df_pts_smq_medication_error))
print("Total de PTs únicos filtrados:", df_pts_smq_medication_error[col_pt].nunique())

# =========================
# 5) Salvar em CSV
# =========================
df_pts_smq_medication_error.to_csv(
    "arquivo_filtrado_smq_erro_de_medicacao_y.csv",
    index=False,
    encoding="utf-8-sig"
)

SUB SELEÇÃO DOSE ERROR E NOM DOSE ERROR

In [ ]:
# =========================
# 4) Criar coluna de subseleção: Dose error / Non dose error
# =========================

pts_subselecao_erro_dose = [
    "Administração de dose não ajustada",
    "Confusão com a dose do produto",
    "Confusão quanto ao regime do produto",
    "Dosagem incorreta administrada",
    "Dosagem não ajustada",
    "Dose adicional administrada",
    "Dose aumentada administrada",
    "Dose de reforço perdida",
    "Dose incorreta",
    "Dose incorreta administrada",
    "Dose incorreta administrada pelo dispositivo",
    "Dose incorreta administrada pelo produto",
    "Dose subterapêutica acidental",
    "Duração incorreta de administração do produto",
    "Erro de cálculo da dose",
    "Erro de titulação de medicamento",
    "Intoxicação acidental",
    "Omissão de dose do medicamento pelo dispositivo",
    "Omissão de dose do produto por erro",
    "Posologia inadequada de administração de produto",
    "Regime posológico incorreto",
    "Superdosagem acidental",
    "Taxa incorreta",
    "Taxa incorreta de administração do medicamento",
    "Titulação de dose do medicamento não realizada",
    "Dose subterapêutica",
    "Dose subterapêutica de radiação",
    "Dose subterapêutica prescrita",
    "Exposição a doses radioativas excessivas",
    "Problema relacionado à omissão de dose do produto",
    "Superdosagem",
    "Superdosagem por prescrição médica"
]

df_pts_smq_medication_error["Subselecao_Erro"] = df_pts_smq_medication_error[col_pt].apply(
    lambda x: "Dose error" if x in pts_subselecao_erro_dose else "Non dose error"
)
print("Total de linhas filtradas:", len(df_pts_smq_medication_error))
print("Total de PTs únicos filtrados:", df_pts_smq_medication_error[col_pt].nunique())

print(
    df_pts_smq_medication_error["Subselecao_Erro"]
    .value_counts()
)
df_pts_smq_medication_error.to_csv(
    "arquivo_filtrado_AGRUPADO_smq_erro_de_medicacao_y.csv",
    index=False,
    encoding="utf-8-sig"
)

Salvar só erro de dose

In [ ]:
# =========================
# 5) Filtrar apenas Dose error
# =========================

df_dose_error = df_pts_smq_medication_error[
    df_pts_smq_medication_error["Subselecao_Erro"] == "Dose error"
].copy()

print("Total de linhas Dose error:", len(df_dose_error))
print("Total de PTs únicos Dose error:", df_dose_error[col_pt].nunique())

# =========================
# 6) Salvar CSV apenas com Dose error
# =========================

df_dose_error.to_csv(
    "arquivo_filtrado_APENAS_dose_error_y.csv",
    index=False,
    encoding="utf-8-sig"
)